# SmartPDS — Layer 2: LP Allocation Optimizer
**Team Kahunas | DevsHouse 26 | Open Innovation**

Consumes the village-level demand forecast from Layer 1 and computes the provably optimal grain allocation across 3 warehouses to 50 villages. The objective minimizes a weighted sum of shortage penalty and transport cost simultaneously using Google OR-Tools Linear Programming.

**Input:** `village_demand_forecast.csv`, `villages.csv` (from Layer 1 outputs)

**Output:** `allocation_plan.csv` (input for Layer 3 — Fraud Detection)

**Pipeline:** Forecast + Village Coords -> Warehouse Setup -> Distance Matrix -> LP Formulation -> Solve -> Analysis -> Export

## 0. Dependencies

In [ ]:
!pip install ortools pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import time

from ortools.linear_solver import pywraplp

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f5f5f5',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Load Layer 1 Outputs

Reads the demand forecast and village geography files produced by the Layer 1 notebook. Both files must be in the same working directory.

In [ ]:
forecast_df = pd.read_csv('village_demand_forecast.csv')
villages_df = pd.read_csv('villages.csv')

print(f'Forecast records : {len(forecast_df)}')
print(f'Columns          : {list(forecast_df.columns)}')
print()
print(f'Villages records : {len(villages_df)}')
print(f'Columns          : {list(villages_df.columns)}')
print()
print('Forecast preview:')
forecast_df.head()

In [ ]:
# Merge village coordinates into forecast
# villages.csv must have lat, lon columns — adjust column names if different
coord_cols = ['village_id', 'lat', 'lon', 'district']
available  = [c for c in coord_cols if c in villages_df.columns]

df = forecast_df.merge(villages_df[available], on='village_id', how='left')

# If coordinates are missing, generate realistic Tamil Nadu coordinates
# Tamil Nadu bounding box: lat 8.0-13.5, lon 76.9-80.3
if 'lat' not in df.columns or df['lat'].isnull().all():
    print('Coordinates not found in villages.csv — generating realistic Tamil Nadu coords')
    np.random.seed(42)
    df['lat'] = np.random.uniform(8.0, 13.5, len(df))
    df['lon'] = np.random.uniform(76.9, 80.3, len(df))
else:
    print('Coordinates loaded from villages.csv')

df.reset_index(drop=True, inplace=True)
print(f'\nMerged dataframe shape: {df.shape}')
print(f'Null coordinates: {df[["lat","lon"]].isnull().sum().sum()}')
df.head()

## 2. Warehouse Configuration

Three warehouses are positioned across Tamil Nadu — one each in the northern, central, and southern regions — reflecting FCI depot distribution. Stock levels are set proportional to regional beneficiary populations.

In [ ]:
WAREHOUSES = [
    {
        'warehouse_id':   'WH_NORTH',
        'name':           'Chennai Central Depot',
        'lat':            13.0827,
        'lon':            80.2707,
        'stock_kg':       80000,
        'region':         'North'
    },
    {
        'warehouse_id':   'WH_CENTRAL',
        'name':           'Tiruchirappalli Depot',
        'lat':            10.7905,
        'lon':            78.7047,
        'stock_kg':       65000,
        'region':         'Central'
    },
    {
        'warehouse_id':   'WH_SOUTH',
        'name':           'Madurai South Depot',
        'lat':             9.9252,
        'lon':            78.1198,
        'stock_kg':       55000,
        'region':         'South'
    },
]

wh_df = pd.DataFrame(WAREHOUSES)

total_stock    = wh_df['stock_kg'].sum()
total_demand   = df['forecasted_demand_kg'].sum()
coverage_ratio = total_stock / total_demand

print('Warehouse configuration:')
print(wh_df[['warehouse_id','name','region','stock_kg']].to_string(index=False))
print()
print(f'Total warehouse stock  : {total_stock:,.0f} kg')
print(f'Total forecasted demand: {total_demand:,.0f} kg')
print(f'Stock / Demand ratio   : {coverage_ratio:.3f} ({coverage_ratio*100:.1f}% covered)')

## 3. Distance Matrix

Haversine distances (km) are computed between every warehouse and every village. Distance is used as the transport cost coefficient in the LP objective — allocating from a closer warehouse is cheaper.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R    = 6371.0
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a    = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))


n_warehouses = len(WAREHOUSES)
n_villages   = len(df)

dist_matrix = np.zeros((n_warehouses, n_villages))

for w_idx, wh in enumerate(WAREHOUSES):
    for v_idx, row in df.iterrows():
        dist_matrix[w_idx, v_idx] = haversine_km(
            wh['lat'], wh['lon'],
            row['lat'], row['lon']
        )

print(f'Distance matrix shape: {dist_matrix.shape}  ({n_warehouses} warehouses x {n_villages} villages)')
print()
print('Average distances per warehouse (km):')
for w_idx, wh in enumerate(WAREHOUSES):
    print(f'  {wh["warehouse_id"]:12s} : avg={dist_matrix[w_idx].mean():.1f} km   '
          f'min={dist_matrix[w_idx].min():.1f} km   '
          f'max={dist_matrix[w_idx].max():.1f} km')

## 4. Shortage Risk Weights

High-risk villages receive a 3x penalty weight in the objective, ensuring the optimizer prioritizes covering them before lower-risk villages when stock is constrained.

In [ ]:
RISK_WEIGHTS = {'High': 3.0, 'Medium': 2.0, 'Low': 1.0}

df['risk_weight'] = df['shortage_risk'].map(RISK_WEIGHTS).fillna(1.0)

print('Shortage risk distribution:')
print(df['shortage_risk'].value_counts().to_string())
print()
print('Risk weights applied:')
for risk, weight in RISK_WEIGHTS.items():
    count = (df['shortage_risk'] == risk).sum()
    print(f'  {risk:8s} : weight={weight:.1f}  villages={count}')

## 5. LP Formulation and Solve

Decision variables: `x[w][v]` = kg allocated from warehouse `w` to village `v`.

Constraints:
- Supply: total allocated from each warehouse cannot exceed its stock
- Demand cap: each village receives at most its forecasted demand
- Non-negativity: all allocations >= 0

Objective minimizes: `ALPHA * shortage_penalty + BETA * transport_cost`

`ALPHA` and `BETA` control the tradeoff. Default: 1000 vs 1 strongly prioritizes coverage over distance.

In [ ]:
ALPHA = 1000.0  # shortage penalty weight
BETA  = 1.0     # transport cost weight (per km per kg)

demands  = df['forecasted_demand_kg'].values
stocks   = wh_df['stock_kg'].values
weights  = df['risk_weight'].values

solver = pywraplp.Solver.CreateSolver('GLOP')
if not solver:
    raise RuntimeError('GLOP solver not available. Verify ortools installation.')

# Decision variables: x[w][v]
x = {}
for w in range(n_warehouses):
    for v in range(n_villages):
        x[w, v] = solver.NumVar(0.0, demands[v], f'x_{w}_{v}')

# Supply constraints
for w in range(n_warehouses):
    constraint = solver.Constraint(0, stocks[w], f'supply_{w}')
    for v in range(n_villages):
        constraint.SetCoefficient(x[w, v], 1.0)

# Demand cap constraints (village receives at most its forecasted demand)
for v in range(n_villages):
    constraint = solver.Constraint(0, demands[v], f'demand_cap_{v}')
    for w in range(n_warehouses):
        constraint.SetCoefficient(x[w, v], 1.0)

# Objective: minimize (alpha * shortage_penalty) + (beta * transport_cost)
# shortage_penalty for village v = weight[v] * (demand[v] - sum_w x[w,v])
# transport_cost   for pair (w,v) = dist[w,v] * x[w,v]
objective = solver.Objective()

for v in range(n_villages):
    for w in range(n_warehouses):
        transport_coeff = BETA  * dist_matrix[w, v]
        shortage_coeff  = -ALPHA * weights[v]   # negative because maximizing allocation = minimizing shortage
        objective.SetCoefficient(x[w, v], transport_coeff + shortage_coeff)

objective.SetMinimization()

print(f'LP formulation complete')
print(f'  Variables   : {solver.NumVariables()}')
print(f'  Constraints : {solver.NumConstraints()}')
print(f'  Alpha (shortage weight) : {ALPHA}')
print(f'  Beta  (transport weight): {BETA}')
print()
print('Solving...')

t0     = time.time()
status = solver.Solve()
elapsed = time.time() - t0

STATUS_MAP = {
    pywraplp.Solver.OPTIMAL:    'OPTIMAL',
    pywraplp.Solver.FEASIBLE:   'FEASIBLE',
    pywraplp.Solver.INFEASIBLE: 'INFEASIBLE',
    pywraplp.Solver.UNBOUNDED:  'UNBOUNDED',
    pywraplp.Solver.ABNORMAL:   'ABNORMAL',
}

print(f'Solver status : {STATUS_MAP.get(status, "UNKNOWN")}')
print(f'Solve time    : {elapsed:.4f} seconds')

if status not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
    raise RuntimeError(f'LP did not find a solution. Status: {STATUS_MAP.get(status)}')

## 6. Extract and Structure Results

In [ ]:
allocation_records = []

for v in range(n_villages):
    row           = df.iloc[v]
    total_alloc   = 0.0
    primary_wh    = None
    primary_alloc = 0.0

    wh_breakdown = {}
    for w in range(n_warehouses):
        alloc = x[w, v].solution_value()
        wh_breakdown[WAREHOUSES[w]['warehouse_id']] = round(alloc, 2)
        total_alloc += alloc
        if alloc > primary_alloc:
            primary_alloc = alloc
            primary_wh    = WAREHOUSES[w]['warehouse_id']

    shortage_gap = max(0, row['forecasted_demand_kg'] - total_alloc)
    coverage_pct = (total_alloc / row['forecasted_demand_kg'] * 100) if row['forecasted_demand_kg'] > 0 else 0

    rec = {
        'village_id':           row['village_id'],
        'forecasted_demand_kg': round(row['forecasted_demand_kg'], 1),
        'allocated_kg':         round(total_alloc, 1),
        'shortage_gap_kg':      round(shortage_gap, 1),
        'coverage_pct':         round(coverage_pct, 1),
        'primary_warehouse':    primary_wh,
        'shortage_risk':        row['shortage_risk'],
        'risk_weight':          row['risk_weight'],
    }
    for wh_id, alloc in wh_breakdown.items():
        rec[f'alloc_{wh_id}'] = alloc

    if 'village_name' in row.index:
        rec['village_name'] = row['village_name']
    if 'district' in row.index or 'district_x' in row.index:
        rec['district'] = row.get('district', row.get('district_x', ''))
    if 'bpl_count' in row.index:
        rec['bpl_count'] = row['bpl_count']

    allocation_records.append(rec)

alloc_df = pd.DataFrame(allocation_records).sort_values('shortage_gap_kg', ascending=False).reset_index(drop=True)

print(f'Allocation complete')
print(f'  Total allocated  : {alloc_df["allocated_kg"].sum():,.1f} kg')
print(f'  Total demand     : {alloc_df["forecasted_demand_kg"].sum():,.1f} kg')
print(f'  Total shortage   : {alloc_df["shortage_gap_kg"].sum():,.1f} kg')
print(f'  Villages fully covered  : {(alloc_df["coverage_pct"] >= 99.9).sum()}')
print(f'  Villages partially covered: {((alloc_df["coverage_pct"] > 0) & (alloc_df["coverage_pct"] < 99.9)).sum()}')
print(f'  Villages with zero alloc: {(alloc_df["allocated_kg"] == 0).sum()}')
print()
alloc_df.head(10)

## 7. Warehouse Utilization Analysis

In [ ]:
wh_util = []
for w_idx, wh in enumerate(WAREHOUSES):
    used = sum(x[w_idx, v].solution_value() for v in range(n_villages))
    wh_util.append({
        'warehouse_id': wh['warehouse_id'],
        'name':         wh['name'],
        'region':       wh['region'],
        'stock_kg':     wh['stock_kg'],
        'used_kg':      round(used, 1),
        'remaining_kg': round(wh['stock_kg'] - used, 1),
        'utilization_pct': round(used / wh['stock_kg'] * 100, 1),
    })

wh_util_df = pd.DataFrame(wh_util)

print('Warehouse utilization:')
print(wh_util_df[['warehouse_id','name','stock_kg','used_kg','remaining_kg','utilization_pct']].to_string(index=False))

## 8. Baseline Comparison

Compares the LP solution against a naive proportional baseline (allocate proportionally to demand, ignoring distance and risk weights). Demonstrates the improvement from optimization.

In [ ]:
# Naive baseline: allocate proportionally from total stock, ignoring warehouse geography
naive_total_stock   = total_stock
naive_alloc         = np.minimum(demands, demands / demands.sum() * naive_total_stock)
naive_shortage      = np.maximum(0, demands - naive_alloc)
naive_shortage_total = naive_shortage.sum()

# Weighted shortage (penalizes high-risk villages more)
lp_weighted_shortage    = (alloc_df['shortage_gap_kg'].values * alloc_df['risk_weight'].values).sum()
naive_weighted_shortage = (naive_shortage * weights).sum()

improvement_pct = (naive_weighted_shortage - lp_weighted_shortage) / naive_weighted_shortage * 100

print('Comparison: LP Optimizer vs Naive Proportional Baseline')
print('-' * 56)
print(f'  {"Metric":<35} {"Naive":>8} {"LP":>8}')
print('-' * 56)
print(f'  {"Total shortage (kg)":<35} {naive_shortage_total:>8.1f} {alloc_df["shortage_gap_kg"].sum():>8.1f}')
print(f'  {"Weighted shortage score":<35} {naive_weighted_shortage:>8.1f} {lp_weighted_shortage:>8.1f}')
print(f'  {"High-risk villages fully covered":<35} {int((naive_alloc[df["shortage_risk"]=="High"] >= demands[df["shortage_risk"]=="High"]).sum()):>8} {int((alloc_df[alloc_df["shortage_risk"]=="High"]["coverage_pct"] >= 99.9).sum()):>8}')
print('-' * 56)
print(f'  Weighted shortage reduction: {improvement_pct:.1f}%')

## 9. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 13))
fig.suptitle('SmartPDS — LP Allocation Optimizer Results', fontsize=15, fontweight='bold')

# Plot 1: Allocated vs Forecasted for top 20 villages by demand
ax = axes[0, 0]
top20 = alloc_df.head(20)
label_col = 'village_name' if 'village_name' in top20.columns else 'village_id'
labels = top20[label_col].astype(str).str[-10:]
x_pos  = np.arange(len(top20))
width  = 0.38

ax.barh(x_pos + width/2, top20['forecasted_demand_kg'], width, label='Forecasted demand', color='#2c7bb6', alpha=0.85)
ax.barh(x_pos - width/2, top20['allocated_kg'],         width, label='LP allocated',      color='#1a9641', alpha=0.85)
ax.set_yticks(x_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Quantity (kg)')
ax.set_title('Forecasted vs Allocated — Top 20 Villages')
ax.legend(fontsize=9)
ax.invert_yaxis()

# Plot 2: Warehouse utilization
ax = axes[0, 1]
wh_names = wh_util_df['warehouse_id']
x_pos2   = np.arange(len(wh_util_df))
width2   = 0.38
ax.bar(x_pos2 + width2/2, wh_util_df['stock_kg'], width2, label='Available stock', color='#2c7bb6', alpha=0.8)
ax.bar(x_pos2 - width2/2, wh_util_df['used_kg'],  width2, label='Allocated',       color='#1a9641', alpha=0.8)
ax.set_xticks(x_pos2)
ax.set_xticklabels(wh_names, rotation=10)
ax.set_ylabel('Quantity (kg)')
ax.set_title('Warehouse Stock vs Allocated')
ax.legend()

for i, row in wh_util_df.iterrows():
    ax.text(i - width2/2, row['used_kg'] + 200, f"{row['utilization_pct']}%",
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 3: Coverage distribution by shortage risk
ax = axes[1, 0]
risk_order  = ['High', 'Medium', 'Low']
risk_colors = {'High': '#d7191c', 'Medium': '#fdae61', 'Low': '#1a9641'}
for risk in risk_order:
    subset = alloc_df[alloc_df['shortage_risk'] == risk]['coverage_pct']
    if len(subset) > 0:
        ax.hist(subset, bins=15, alpha=0.7, label=f'{risk} ({len(subset)} villages)',
                color=risk_colors[risk], edgecolor='white')
ax.axvline(100, color='black', linestyle='--', lw=1.2, label='Full coverage')
ax.set_xlabel('Coverage (%)')
ax.set_ylabel('Number of villages')
ax.set_title('Coverage Distribution by Shortage Risk')
ax.legend(fontsize=9)

# Plot 4: Shortage gap by village (top 20 worst)
ax = axes[1, 1]
worst20     = alloc_df[alloc_df['shortage_gap_kg'] > 0].head(20)
label_col2  = 'village_name' if 'village_name' in worst20.columns else 'village_id'
bar_colors  = [risk_colors.get(str(r), '#2c7bb6') for r in worst20['shortage_risk']]
ax.barh(worst20[label_col2].astype(str).str[-10:], worst20['shortage_gap_kg'],
        color=bar_colors, edgecolor='white')
ax.set_xlabel('Shortage gap (kg)')
ax.set_title('Top 20 Villages by Unmet Demand')
ax.invert_yaxis()
legend_patches = [mpatches.Patch(color=c, label=l) for l, c in risk_colors.items()]
ax.legend(handles=legend_patches, title='Risk', fontsize=9)

plt.tight_layout()
plt.savefig('lp_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# District-level summary if district column is available
if 'district' in alloc_df.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    fig.suptitle('SmartPDS — District-Level Allocation Summary', fontsize=13, fontweight='bold')

    dist_summary = (
        alloc_df.groupby('district')
        .agg(total_demand=('forecasted_demand_kg', 'sum'),
             total_allocated=('allocated_kg', 'sum'),
             total_shortage=('shortage_gap_kg', 'sum'))
        .sort_values('total_demand', ascending=True)
        .reset_index()
    )

    x_pos3 = np.arange(len(dist_summary))
    w3     = 0.3
    ax.barh(x_pos3 + w3/2, dist_summary['total_demand'],    w3, label='Demand',    color='#2c7bb6', alpha=0.85)
    ax.barh(x_pos3 - w3/2, dist_summary['total_allocated'], w3, label='Allocated', color='#1a9641', alpha=0.85)
    ax.set_yticks(x_pos3)
    ax.set_yticklabels(dist_summary['district'], fontsize=9)
    ax.set_xlabel('Total quantity (kg)')
    ax.set_title('District-Level Demand vs Allocation')
    ax.legend()

    plt.tight_layout()
    plt.savefig('district_allocation.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('District column not available — skipping district plot')

## 10. Full Allocation Summary Table

In [ ]:
display_cols = ['village_id', 'forecasted_demand_kg', 'allocated_kg',
                'shortage_gap_kg', 'coverage_pct', 'primary_warehouse', 'shortage_risk']

if 'village_name' in alloc_df.columns:
    display_cols.insert(1, 'village_name')
if 'district' in alloc_df.columns:
    display_cols.insert(2, 'district')

print('Full allocation plan (sorted by shortage gap):')
print(alloc_df[display_cols].to_string(index=False))

## 11. Export Outputs

The allocation plan CSV is the direct input to Layer 3 — Isolation Forest Fraud Detection. The fraud detector uses the gap between `allocated_kg` and actual collected quantity to flag anomalies.

In [ ]:
export_cols = ['village_id', 'forecasted_demand_kg', 'allocated_kg',
               'shortage_gap_kg', 'coverage_pct', 'primary_warehouse', 'shortage_risk']

if 'village_name' in alloc_df.columns: export_cols.insert(1, 'village_name')
if 'district'     in alloc_df.columns: export_cols.insert(2, 'district')
if 'bpl_count'    in alloc_df.columns: export_cols.append('bpl_count')

for wh in WAREHOUSES:
    col = f'alloc_{wh["warehouse_id"]}'
    if col in alloc_df.columns:
        export_cols.append(col)

alloc_df[export_cols].to_csv('allocation_plan.csv', index=False)
wh_util_df.to_csv('warehouse_utilization.csv', index=False)

print('Outputs written:')
print('  allocation_plan.csv        — input for Layer 3 Fraud Detection')
print('  warehouse_utilization.csv  — warehouse stock usage report')
print()
print('LP Optimizer Summary')
print('-' * 48)
print(f'  Solver              : GLOP (OR-Tools)')
print(f'  Status              : {STATUS_MAP.get(status)}')
print(f'  Solve time          : {elapsed:.4f} seconds')
print(f'  Variables           : {solver.NumVariables()}')
print(f'  Constraints         : {solver.NumConstraints()}')
print(f'  Warehouses          : {n_warehouses}')
print(f'  Villages            : {n_villages}')
print(f'  Total stock         : {total_stock:,.0f} kg')
print(f'  Total demand        : {total_demand:,.0f} kg')
print(f'  Total allocated     : {alloc_df["allocated_kg"].sum():,.1f} kg')
print(f'  Total shortage      : {alloc_df["shortage_gap_kg"].sum():,.1f} kg')
print(f'  Coverage ratio      : {alloc_df["allocated_kg"].sum()/total_demand*100:.1f}%')
print(f'  Weighted improvement: {improvement_pct:.1f}% over naive baseline')
print('-' * 48)